# Stitch viewer: tiles, design, contours and design errors, before and after stitching

Runs the pipeline on the chosen tiles and opens one interactive window (pyqtgraph, OpenGL):

1. **Per tile** (`process_tile`): load the SEM image, detect contacts (Otsu), refine the contours
   to the maximum gradient, read the design contacts from the tile's `.oas` in the tone (normal /
   reversed) that matches the SEM contacts. Tiles where neither tone matches are flagged.
2. **Stitching**, translation per tile, failed overlaps and unstitched tiles listed:
   the SEM tiles (`stitch_tiles`) and, the same way, the per-tile design files (`stitch_design`),
   which can be offset against each other.
3. **Design errors** (`design_errors`) for each placement (`placement_corrections`): nominal,
   stitched (corrections average 0), stitched with the first tile fixed. Raw error SEM − design,
   no affine removed yet. Tiles that could not be measured are listed with the reason.
4. **Viewer** (`StitchViewer`): left, SEM images, design contours/centres, Otsu and refined
   contours/centres, overlap boxes and overlap outliers; right, the error map of the selected
   placement, sharing the zoom. Layers and single tiles can be hidden on the right.

The window blocks this notebook until it is closed. Positions in µm, errors in nm.

In [ ]:
DATA_DIR = r"A:\Stitch_monitor\In_cell_without_PTM_20260914\260910_1143_Norm"
TILES = None              # None = all tiles, or a list of indices in read_tile_index order, e.g. list(range(10))
LAYER = (100, 0)          # (layer, datatype) of the contacts in the .oas
OTSU_CLASSES = 3          # 2: plain Otsu; 3: hole | background | bright band, threshold hole | background (wide bands)
SEARCH_PX = 5.0           # refinement searches this many pixels in/out along the contour normal
DESIGN_MATCH_NM = 40.0    # pair design and SEM contacts closer than this (tone check and design errors)
MIN_MATCH_FRACTION = 0.5  # a design tone is accepted if at least this fraction of SEM contacts match it
STITCH_REFINED = True     # stitch with the refined centres (False: Otsu centres)
MAX_MATCH_NM = 25.0       # SEM overlaps: pair contacts closer than this, the largest B - A stage offset (< pitch / 2)
DESIGN_MAX_MATCH_NM = 25.0  # design overlaps: the largest offset between neighbouring .oas files (< pitch / 2)
OUTLIER_FACTOR = 3.0      # overlap outlier if residual > this x the median residual
MIN_MATCHED = 5           # pairs with fewer matched contacts are not used (reported below)
GROUP_RADIUS_NM = 5.0     # observations from different tiles whose stitched design positions are this close = one contact
SPREAD_FLAG_NM = 2.0      # flag merged contacts whose SEM observations differ by more than this
ARROW_SCALE = 10          # error-map lines are drawn this many times longer than the error
USE_OPENGL = True         # set False if the viewer window stays black
RANSAC_PLACEMENT = "mean"   # RANSAC on the merged contacts of this placement: "mean" (stitched) or "first"
RANSAC_THRESHOLD_NM = 0.5   # RANSAC inlier if |G(SEM) - design| < this
RANSAC_SEED = 0             # RANSAC random samples are reproducible for a given seed
RANSAC_DELAY_MS = 300       # RANSAC monitor: pause between steps
ROW_GAP_NM = 10.0           # registration error per row: a new row starts where sorted design y jumps by more than this
MOVING_WINDOW_UM = 40.0     # moving-window affine: window height along y
MOVING_STEP_UM = 5.0        # moving-window affine: the window moves up this much each step

In [ ]:
import time

import numpy as np
import pandas as pd
import pyqtgraph as pg

from affine_ransac.fitting.moving_window import moving_window_affine
from affine_ransac.fitting.ransac import ransac_affine
from affine_ransac.geometry.affine import report_terms
from affine_ransac.io.design import load_layout, read_polygons
from affine_ransac.io.metadata import read_tile_index
from affine_ransac.pipeline import process_tile, stitch_design, stitch_tiles
from affine_ransac.registration import design_errors, error_summary, merge_observations, row_means, row_pitch
from affine_ransac.stitching import placement_corrections
from affine_ransac.view_moving_window import MovingWindowTuner
from affine_ransac.view_pitch import PitchView
from affine_ransac.view_ransac import RansacMonitor
from affine_ransac.view_registration import RegistrationView
from affine_ransac.view_stitch import StitchViewer

## 1. Load the tiles and detect their contacts

In [ ]:
all_tiles = read_tile_index(DATA_DIR)
records = all_tiles if TILES is None else [all_tiles[i] for i in TILES]
tile_ids = [r.tile_id for r in records]

start = time.perf_counter()
tiles = [
    process_tile(r.image_path, r.oas_path, (r.center_x_nm, r.center_y_nm), (r.fov_x_nm, r.fov_y_nm),
                 LAYER, SEARCH_PX, DESIGN_MATCH_NM, MIN_MATCH_FRACTION, OTSU_CLASSES)
    for r in records
]
print(f"{len(tiles)} of {len(all_tiles)} tiles in {time.perf_counter() - start:.1f} s")

per_tile = pd.DataFrame({
    "tile": tile_ids,
    "design": [len(t.design.centers) for t in tiles],
    "otsu": [len(t.otsu.centers) for t in tiles],
    "refined": [len(t.refined.centers) for t in tiles],
    "tone": ["reversed" if t.design.reversed else "normal" for t in tiles],
    "match_fraction": [t.design.match_fraction for t in tiles],
    "design_ok": [t.design.ok for t in tiles],
})
print(f"Design tone: {(per_tile.tone == 'normal').sum()} normal, {(per_tile.tone == 'reversed').sum()} reversed")
flagged = per_tile[~per_tile.design_ok]
if len(flagged):
    print(f"\n!! {len(flagged)} tile(s) FLAGGED: neither design tone matches the SEM (or no .oas):")
    display(flagged.round(2))
per_tile.describe().round(2)

## 2. Stitching: SEM tiles and design files

In [ ]:
centers = np.array([t.center_nm for t in tiles])
fovs = np.array([t.fov_nm for t in tiles])
stitch = stitch_tiles([t.sem_points_nm(STITCH_REFINED) for t in tiles], centers, fovs,
                      MAX_MATCH_NM, OUTLIER_FACTOR, MIN_MATCHED)
design_stitch = stitch_design(tiles, DESIGN_MAX_MATCH_NM, OUTLIER_FACTOR, MIN_MATCHED)

for name, result in (("SEM", stitch), ("Design", design_stitch)):
    failed = [r for r in result.rejected if r.failed]
    print(f"{name}: {len(result.pairs)} overlapping pairs used, {len(failed)} not matched, "
          f"{len(result.rejected) - len(failed)} skipped (fewer than {MIN_MATCHED} contacts in the overlap); "
          f"corrections |x| max {np.nanmax(np.abs(result.corrections[:, 0])):.2f} nm, "
          f"|y| max {np.nanmax(np.abs(result.corrections[:, 1])):.2f} nm")
    if failed or len(result.unplaced):
        print(f"!! {name.upper()} STITCHING INCOMPLETE. Tiles not stitched: {[tile_ids[k] for k in result.unplaced]}")
        display(pd.DataFrame([{"tile_a": tile_ids[r.i], "tile_b": tile_ids[r.j], "contacts_in_overlap": r.in_box,
                               "matched": r.matched} for r in failed]))

pd.DataFrame({
    "tile": tile_ids,
    "sem_correction_x_nm": stitch.corrections[:, 0], "sem_correction_y_nm": stitch.corrections[:, 1],
    "design_correction_x_nm": design_stitch.corrections[:, 0], "design_correction_y_nm": design_stitch.corrections[:, 1],
}).round(2)

## 3. Design errors per placement (raw: SEM − design, no affine removed)

Each tile is matched to its own design, so a contact in an overlap is first observed once per
tile (2 in a strip, up to 4 in a corner). `merge_observations` then turns the observations into one
entry per physical contact: observations from different tiles whose **stitched design** positions
are within `GROUP_RADIUS_NM` are averaged. Their **spread** (largest distance between the SEM
observations) is kept; a large spread means the tiles disagree about that contact.

The common offset depends on the placement, so look at the pattern: a uniform shift, a rotation
(arrows circling) or a scale (arrows pointing outward/inward).

In [ ]:
sem_points = [t.sem_points_nm(STITCH_REFINED) for t in tiles]
design_points = [t.design_points_nm() for t in tiles]
design_ok = [t.design.ok for t in tiles]
observations, merged = {}, {}
for placement, (sem_corrections, design_corrections) in placement_corrections(stitch.corrections,
                                                                               design_stitch.corrections).items():
    observations[placement] = design_errors(sem_points, design_points, sem_corrections, design_corrections,
                                            design_ok, DESIGN_MATCH_NM)
    merged[placement] = merge_observations(observations[placement], design_stitch.corrections, GROUP_RADIUS_NM)

for placement, e in observations.items():
    if e.skipped:
        print(f"!! {placement}: {len(e.skipped)} tile(s) NOT MEASURED (no error computed):")
        display(pd.DataFrame([{"tile": tile_ids[k], "reason": reason} for k, reason in e.skipped.items()]))

m = merged["mean"]
sizes = pd.Series(m.count).value_counts().sort_index()
print(f"{len(observations['mean'].tile)} observations -> {len(m.count)} contacts; "
      f"observations per contact: {({int(k): int(v) for k, v in sizes.items()})}")
overlap = m.count > 1
if overlap.any():
    print(f"Spread of merged overlap contacts (stitched): median {np.median(m.spread_nm[overlap]):.2f} nm, "
          f"max {m.spread_nm.max():.2f} nm; {(m.spread_nm > SPREAD_FLAG_NM).sum()} above {SPREAD_FLAG_NM} nm")
    worst = np.argsort(m.spread_nm)[::-1][:10]
    display(pd.DataFrame({
        "x_um": m.design_nm[worst, 0] / 1000, "y_um": m.design_nm[worst, 1] / 1000,
        "spread_nm": m.spread_nm[worst], "observations": m.count[worst],
        "tiles": [[tile_ids[k] for k in observations["mean"].tile[m.members[g]]] for g in worst],
    }).round(4))

pd.DataFrame({placement: error_summary(e.error_nm) for placement, e in merged.items()}).round(2)

## 4. Viewer (closes with the window)

Left: SEM and design. Right: the error map of the selected placement, one line per contact
(dot = mean design position, line = mean error × `ARROW_SCALE`, adjustable in the panel); merged
overlap contacts can be ringed, magenta rings mark a spread above `SPREAD_FLAG_NM`. Both plots
share zoom and pan.

In [ ]:
design_polygons = [read_polygons(load_layout(r.oas_path), *LAYER) if r.oas_path else [] for r in records]

app = pg.mkQApp("Stitch viewer")
viewer = StitchViewer(tiles, tile_ids, design_polygons, stitch, design_stitch, refined=STITCH_REFINED,
                      use_opengl=USE_OPENGL, errors=merged, arrow_scale=ARROW_SCALE, spread_flag_nm=SPREAD_FLAG_NM)
viewer.show()
app.exec()

## 5. RANSAC global affine (on the merged contacts)

`ransac_affine` finds one affine G for the whole field that maps the stitched SEM centres onto the
design centres: random 3-point samples, the model with the most inliers (residual <
`RANSAC_THRESHOLD_NM`), then a least-squares refit on its inliers, all relative to the design
centroid. Outliers are only left out of the fit; their residuals are kept. The first cell runs it
without a UI; the second opens the live monitor (closes with the window).

**Table** (correction G, SEM → design): Tx, Ty in nm; Mx, My in ppm (= nm per mm of distance);
rotation and orthogonality in degrees. *edge (nm)*: how far that term alone moves the furthest
contact, comparable to the threshold and the residuals.

**Linear plots in the monitor** (raw error SEM − design vs position; opposite sign to G):
dx vs x and dy vs y tilt with a **scale** error; dx vs y and dy vs x tilt in **opposite**
directions for a **rotation** and in the **same** direction for an **orthogonality** (skew)
error; a vertical offset is a **shift**. Green band around the blue model line = inliers (its
width = noise); red points far off = outliers; a curved trend = distortion an affine cannot fit.

In [ ]:
contacts = merged[RANSAC_PLACEMENT]
result = ransac_affine(contacts.sem_nm, contacts.design_nm, RANSAC_THRESHOLD_NM, seed=RANSAC_SEED)

print(f"{result.iterations} search iterations; best sample {result.best_sample.tolist()}")
print(f"Inliers: {result.inliers.sum()} / {len(result.inliers)} ({result.inliers.mean():.1%}), "
      f"reference (design centroid): x {result.reference[0] / 1000:.4f} um, y {result.reference[1] / 1000:.4f} um")
display(pd.DataFrame(report_terms(result.model, contacts.design_nm - result.reference),
                     columns=["term (correction G)", "value", "edge (nm)"]).set_index("term (correction G)"))
print("G = [[a, b, tx], [c, d, ty]] about the reference point:")
print(np.array2string(result.model[:2], precision=9, suppress_small=False))
pd.DataFrame({"inliers": error_summary(result.residuals[result.inliers]),
              "all contacts": error_summary(result.residuals)}).round(3)

In [ ]:
app = pg.mkQApp("RANSAC monitor")
monitor = RansacMonitor(contacts.sem_nm, contacts.design_nm, RANSAC_THRESHOLD_NM, seed=RANSAC_SEED,
                        interval_ms=RANSAC_DELAY_MS, use_opengl=USE_OPENGL)
monitor.show()
app.exec()

## 6. Registration error after the RANSAC affine

The final registration error of every contact: its residual after the global affine G,
G(SEM) − design (`result.residuals`), outliers included. The viewer shows two error maps (dx, dy):
every contact a dot in the jet colour map, rasterized to the screen (contacts sharing a pixel are
averaged when zoomed out), and the mean dx, dy of each row of contacts (rows grouped by design y,
`ROW_GAP_NM`) against the row's y. Closes with the window.

In [ ]:
registration_error = result.residuals  # (N, 2) G(SEM) - design per merged contact, nm (outliers included)
row_y, row_mean, row_count = row_means(contacts.design_nm[:, 1], registration_error, ROW_GAP_NM)
print(f"{len(row_y)} rows, {row_count.min()}..{row_count.max()} contacts per row")
display(pd.DataFrame({"row_y_um": (row_y - result.reference[1]) / 1000, "contacts": row_count,
                      "dx_mean_nm": row_mean[:, 0], "dy_mean_nm": row_mean[:, 1]}).round(3))

app = pg.mkQApp("Registration error")
registration_view = RegistrationView(contacts.design_nm, registration_error, result.reference, ROW_GAP_NM,
                                     use_opengl=USE_OPENGL)
registration_view.show()
app.exec()

## 7. Moving-window affine, compared with the RANSAC affine

Instead of one affine for the whole field, `moving_window_affine` fits a plain least-squares affine
(no RANSAC, all contacts) in a window `MOVING_WINDOW_UM` tall along y, moves the window up by
`MOVING_STEP_UM`, and fits again, from the bottom of the field to the top. Each contact is
corrected by the window whose centre is nearest to its design y, so its residual
G_window(SEM) − design comes from a fit centred on it. Same sign as section 6.

The first cell compares the summaries and the per-row means with the RANSAC result and lists
the affine terms of every window (correction G, as in section 5; *edge* columns left out). The
second opens both registration-error views side by side (same reference point); closes with the
windows.

In [ ]:
moving = moving_window_affine(contacts.sem_nm, contacts.design_nm, MOVING_WINDOW_UM * 1000, MOVING_STEP_UM * 1000)
print(f"{len(moving.centres_nm)} windows, {moving.counts.min()}..{moving.counts.max()} contacts per window")
display(pd.DataFrame({"RANSAC inliers": error_summary(result.residuals[result.inliers]),
                      "RANSAC all contacts": error_summary(result.residuals),
                      "moving window all contacts": error_summary(moving.residuals)}).round(3))

window_terms = pd.DataFrame(
    [{label: value for label, value, _ in report_terms(model, contacts.design_nm[moving.window == k] - reference)}
     for k, (model, reference) in enumerate(zip(moving.models, moving.references))])
window_terms.insert(0, "contacts", moving.counts)
window_terms.insert(0, "centre_y_um", (moving.centres_nm - result.reference[1]) / 1000)
display(window_terms.round(6))

_, moving_row_mean, _ = row_means(contacts.design_nm[:, 1], moving.residuals, ROW_GAP_NM)
pd.DataFrame({"row_y_um": (row_y - result.reference[1]) / 1000, "contacts": row_count,
              "ransac_dx_nm": row_mean[:, 0], "moving_dx_nm": moving_row_mean[:, 0],
              "ransac_dy_nm": row_mean[:, 1], "moving_dy_nm": moving_row_mean[:, 1]}).round(3)

In [ ]:
app = pg.mkQApp("Registration error")
ransac_view = RegistrationView(contacts.design_nm, result.residuals, result.reference, ROW_GAP_NM, use_opengl=USE_OPENGL)
moving_view = RegistrationView(contacts.design_nm, moving.residuals, result.reference, ROW_GAP_NM, use_opengl=USE_OPENGL)
ransac_view.setWindowTitle(f"RANSAC global affine - {ransac_view.windowTitle()}")
moving_view.setWindowTitle(f"Moving window {MOVING_WINDOW_UM:g} um, step {MOVING_STEP_UM:g} um - {moving_view.windowTitle()}")
ransac_view.show()
moving_view.show()
app.exec()

### 7b. Moving-window tuner (closes with the window)

Sliders (or the boxes next to them) set the window height and the step, starting from
`MOVING_WINDOW_UM` and `MOVING_STEP_UM`; the moving-window affine is recomputed when a slider stops
moving. Left: the error maps and the mean error per row (solid: moving window, dashed: RANSAC);
the colour scale is the RANSAC one, so colours compare with section 6. Right: the affine term
chosen in the list for every window against its centre y, with the RANSAC affine taken at the same
points (dashed; its Tx, Ty change along y because a global rotation or scale shifts points
differently in different places). Invalid settings (step > window, a window with < 3 contacts)
are reported above the plots and the last result is kept.

In [ ]:
app = pg.mkQApp("Moving-window tuner")
tuner = MovingWindowTuner(contacts.sem_nm, contacts.design_nm, result, MOVING_WINDOW_UM, MOVING_STEP_UM, ROW_GAP_NM,
                          use_opengl=USE_OPENGL)
tuner.show()
app.exec()
print(f"Last setting: window {tuner.window.value():g} um, step {tuner.step.value():g} um")

## 8. Row-to-row pitch along y

Rows are grouped by design y (`ROW_GAP_NM`); a row's position is the mean y of its contacts and
the pitch is the spacing to the next row up, plotted at the rows' mean design y. Three sets of the
same merged contacts (`RANSAC_PLACEMENT`): the **design**, the **stitched SEM without any affine**
and the **SEM after the RANSAC affine** G. A smooth global affine hardly changes the pitch, so
jumps show local problems: a spike on a tile edge (dotted lines, from the stage centres and FOVs)
points at stitching, a pattern repeating every tile at the per-tile imaging.

The cell lists the summary and the rows where the SEM pitch differs most from the design; the
viewer (top: pitch, bottom: SEM − design) closes with the window. A row whose contacts cover a
different x range than its neighbours picks up rotation × Δx in its mean y (mostly in the set
without affine).

In [ ]:
pitch_sets = {"stitched, no affine": contacts.sem_nm, "after RANSAC affine": contacts.design_nm + result.residuals}
mid_y, design_pitch = row_pitch(contacts.design_nm[:, 1], contacts.design_nm[:, 1], ROW_GAP_NM)
pitch_table = pd.DataFrame({"row_y_um": (mid_y - result.reference[1]) / 1000, "design_nm": design_pitch})
for name, points in pitch_sets.items():
    pitch_table[f"{name} - design (nm)"] = row_pitch(contacts.design_nm[:, 1], points[:, 1], ROW_GAP_NM)[1] - design_pitch
display(pitch_table.describe().round(3))
worst = pitch_table.iloc[:, 2:].abs().max(axis=1).nlargest(10).index
display(pitch_table.loc[worst].round(3))

tile_edges_y = np.unique(np.round(np.concatenate([centers[:, 1] - fovs[:, 1] / 2, centers[:, 1] + fovs[:, 1] / 2])))
app = pg.mkQApp("Row pitch")
pitch_view = PitchView(contacts.design_nm, pitch_sets, result.reference, ROW_GAP_NM, tile_edges_y, use_opengl=USE_OPENGL)
pitch_view.show()
app.exec()